# Full Injection → Detection → Completeness Pipeline (GalSim)

This notebook runs the complete star cluster injection and recovery pipeline
using **GalSim** directly for profile generation and PSF convolution,
avoiding the `inject_from_catalog()` signature issues.

1. **Load data** from Butler (RSP `deepCoadd`)
2. **Configure** injection parameters
3. **Generate catalog** via the pipeline helper
4. **Inject clusters** using GalSim profiles + PSF convolution
5. **Visualize** injection (summary plots & stamp grids)
6. **Run detection** on the injected image
7. **Show detection results** including MCI diagram
8. **Match** detections to injections (retrieval)
9. **Plot completeness curves** as a function of magnitude

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure pipeline is on the path
pipeline_path = os.path.expanduser('~/WORK/INJECT/star-cluster-injection-pipeline')
if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)

import galsim
from lsst.daf.butler import Butler

from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.light_profiles import mag_to_flux
from src.visualization import (
    plot_injection_summary,
    plot_stamp_grid,
    plot_postage_stamps,
)
from src.detection import run_cluster_detection, plot_detection_diagnostic, plot_mci_diagram
from src.retrieval import ClusterRetrieval

plt.style.use('tableau-colorblind10')
%matplotlib inline
print('All imports successful!')

---
## 1 · Load Data from Butler

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')

data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd = butler.get('deepCoadd', dataId=data_id)

# Work on a manageable cutout for the demo
CUTOUT = 1500  # pixels – increase for production runs
image_full = coadd.image.array.copy()
image = image_full[:CUTOUT, :CUTOUT].copy()

print(f'Full coadd shape : {image_full.shape}')
print(f'Cutout shape     : {image.shape}')

fig, ax = plt.subplots(figsize=(7, 7))
vmin, vmax = np.percentile(image, [1, 99])
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title('Input deepCoadd cutout')
plt.show()

---
## 2 · Configure Injection Parameters

In [ ]:
cluster_cfg = ClusterConfig(
    profile_type='king',        # 'king', 'plummer', 'eff', 'sersic'
    method='smooth',            # 'smooth' or 'discrete'
    mag_min=20.0,
    mag_max=26.0,
    r_half_min=2.0,             # pixels
    r_half_max=10.0,
    concentration_min=5.0,      # King concentration c = r_t / r_c
    concentration_max=30.0,
)

config = InjectionConfig(
    run_name='demo_galsim_pipeline',
    n_clusters=100,
    seed=42,
    edge_buffer=50,
    add_noise=True,
    use_actual_psf=False,       # we build the PSF in GalSim below
    save_injected_image=False,
    cluster_config=cluster_cfg,
    tract=data_id['tract'],
    patch=data_id['patch'],
    band=data_id['band'],
    output_dir='outputs/demo_galsim_pipeline',
)

PSF_FWHM = 3.5   # pixels – used for the GalSim Gaussian PSF
ZERO_POINT = 27.0 # photometric zero-point
PIXEL_SCALE = 0.2 # arcsec/pixel (LSST default)

print(config)

---
## 3 · Generate Injection Catalog

We use the pipeline helper to generate the random catalog, but we will
handle the actual injection ourselves with GalSim.

In [ ]:
pipe = InjectionPipeline(config)
pipe.load_data(image=image)

catalog = pipe.generate_catalog()
print(f'Generated catalog with {len(catalog)} clusters')
print(f'First entry: {catalog[0]}')

---
## 4 · Inject Clusters with GalSim

For each catalog entry we:
1. Build a GalSim surface-brightness profile (King, Plummer, Sersic, or EFF)
2. Convolve it with a Gaussian PSF
3. Draw the stamp and add it to the image

In [ ]:
def make_galsim_profile(entry, pixel_scale=PIXEL_SCALE):
    """
    Build a GalSim object for one catalog entry.

    All sizes in the catalog are in **pixels**; GalSim wants arcsec,
    so we multiply by pixel_scale.
    """
    r_half_arcsec = entry['r_half'] * pixel_scale
    profile_type  = entry.get('profile_type', 'king').lower()
    conc          = entry.get('concentration', 10.0)

    if profile_type == 'king':
        # GalSim King takes concentration = r_t / r_0 and half_light_radius
        prof = galsim.King(
            concentration=conc,
            half_light_radius=r_half_arcsec,
        )

    elif profile_type == 'plummer':
        # Plummer is Moffat with beta=2
        # half_light_radius for Moffat
        prof = galsim.Moffat(
            beta=2.0,
            half_light_radius=r_half_arcsec,
        )

    elif profile_type == 'sersic':
        n = conc if conc > 0.3 else 1.0  # concentration stores sersic_n
        prof = galsim.Sersic(
            n=n,
            half_light_radius=r_half_arcsec,
        )

    elif profile_type == 'eff':
        # EFF (Elson, Fall & Freeman) is Moffat with beta = gamma/2
        gamma = conc if conc > 2.01 else 2.5
        prof = galsim.Moffat(
            beta=gamma / 2.0,
            half_light_radius=r_half_arcsec,
        )

    else:
        raise ValueError(f'Unknown profile type: {profile_type}')

    # Set flux from magnitude
    total_flux = mag_to_flux(entry['magnitude'], zero_point=ZERO_POINT)
    prof = prof.withFlux(total_flux)

    return prof

In [ ]:
def inject_clusters_galsim(image, catalog, psf_fwhm, pixel_scale=PIXEL_SCALE,
                           add_noise=True, rng_seed=42):
    """
    Inject all catalog entries into *image* using GalSim.

    Returns
    -------
    injected_image : 2-D ndarray
        Copy of the input image with clusters added.
    injection_info : list[dict]
        One dict per cluster with metadata + 'stamp' array.
    """
    ny, nx = image.shape
    injected = image.copy()
    rng = galsim.BaseDeviate(rng_seed)

    # Build PSF (Gaussian, FWHM in arcsec)
    psf = galsim.Gaussian(fwhm=psf_fwhm * pixel_scale)

    injection_info = []

    for entry in catalog:
        cx = int(round(entry['x']))
        cy = int(round(entry['y']))

        # Build profile and convolve with PSF
        cluster_prof = make_galsim_profile(entry, pixel_scale)
        final = galsim.Convolve([cluster_prof, psf])

        # Determine stamp size (auto from GalSim, capped to avoid huge stamps)
        try:
            stamp = final.drawImage(scale=pixel_scale, method='auto')
        except galsim.errors.GalSimError:
            # If auto-sizing fails, use a fixed large stamp
            stamp_size = int(min(max(20 * entry['r_half'], 64), 256))
            if stamp_size % 2 == 0:
                stamp_size += 1
            stamp = final.drawImage(nx=stamp_size, ny=stamp_size,
                                    scale=pixel_scale, method='auto')

        stamp_arr = stamp.array
        sh, sw = stamp_arr.shape

        # Compute bounds in the image
        y0 = cy - sh // 2
        x0 = cx - sw // 2
        y1 = y0 + sh
        x1 = x0 + sw

        # Clip to image boundaries
        iy0 = max(y0, 0)
        ix0 = max(x0, 0)
        iy1 = min(y1, ny)
        ix1 = min(x1, nx)

        sy0 = iy0 - y0
        sx0 = ix0 - x0
        sy1 = sh - (y1 - iy1)
        sx1 = sw - (x1 - ix1)

        if iy0 >= iy1 or ix0 >= ix1:
            continue  # completely outside image

        injected[iy0:iy1, ix0:ix1] += stamp_arr[sy0:sy1, sx0:sx1]

        info = dict(entry)  # copy catalog entry
        info['stamp'] = stamp_arr
        info['stamp_flux'] = float(stamp_arr.sum())
        injection_info.append(info)

    # Optionally add Poisson noise on the injected sources
    if add_noise:
        diff = injected - image
        noise = np.sqrt(np.clip(diff, 0, None))
        rng_np = np.random.default_rng(rng_seed)
        injected += rng_np.normal(0, np.where(noise > 0, noise, 1e-10))

    return injected, injection_info

In [ ]:
print(f'Injecting {len(catalog)} clusters with GalSim ...')
print(f'  Profile : {config.cluster_config.profile_type}')
print(f'  PSF FWHM: {PSF_FWHM} pixels')

injected_image, injection_info = inject_clusters_galsim(
    image, catalog,
    psf_fwhm=PSF_FWHM,
    pixel_scale=PIXEL_SCALE,
    add_noise=config.add_noise,
    rng_seed=config.seed,
)

print(f'Successfully injected {len(injection_info)} / {len(catalog)} clusters.')

---
## 5 · Visualize the Injection

### 5a – Quick before / after comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = np.percentile(image, [1, 99])
kw = dict(cmap='gray', origin='lower', vmin=vmin, vmax=vmax)

axes[0].imshow(image, **kw)
axes[0].set_title('Original')

axes[1].imshow(injected_image, **kw)
axes[1].set_title('Injected')

diff = injected_image - image
dv = np.percentile(np.abs(diff), 99)
axes[2].imshow(diff, cmap='RdBu_r', origin='lower', vmin=-dv, vmax=dv)
axes[2].set_title('Difference')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### 5b – Summary plot (if the helper works with injection_info dicts)

In [ ]:
os.makedirs(config.output_dir, exist_ok=True)

try:
    plot_injection_summary(
        image,
        injected_image,
        injection_info,
        save_path=os.path.join(config.output_dir, 'injection_summary.png'),
        show=True,
    )
except Exception as e:
    print(f'plot_injection_summary skipped: {e}')

### 5c – Stamp grid

In [ ]:
try:
    plot_stamp_grid(
        injection_info,
        n_cols=5,
        n_rows=4,
        save_path=os.path.join(config.output_dir, 'stamp_grid.png'),
        show=True,
    )
except Exception as e:
    # Fallback: show stamps manually
    print(f'plot_stamp_grid skipped ({e}), showing manually ...')
    fig, axes = plt.subplots(4, 5, figsize=(14, 11))
    for ax, info in zip(axes.flat, injection_info[:20]):
        s = info.get('stamp')
        if s is not None:
            ax.imshow(s, origin='lower', cmap='magma')
            ax.set_title(f"m={info['magnitude']:.1f}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    plt.suptitle('Injected Cluster Stamps', fontsize=14)
    plt.tight_layout()
    plt.show()

### 5d – Injected parameter distributions

In [ ]:
mags  = [info['magnitude'] for info in injection_info]
rhs   = [info['r_half'] for info in injection_info]
concs = [info.get('concentration', np.nan) for info in injection_info]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(mags, bins=20, color='steelblue', edgecolor='k')
axes[0].set_xlabel('Magnitude'); axes[0].set_ylabel('N')

axes[1].hist(rhs, bins=20, color='darkorange', edgecolor='k')
axes[1].set_xlabel('r_half (pix)')

axes[2].hist(concs, bins=20, color='seagreen', edgecolor='k')
axes[2].set_xlabel('Concentration')

plt.suptitle('Injected Cluster Parameters', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6 · Run Detection on the Injected Image

In [ ]:
detections = run_cluster_detection(
    image=injected_image,
    psf_fwhm=PSF_FWHM,
    threshold_sigma=5.0,
    use_multiscale=True,
    use_mci=True,
    mci_max=0.85,
)

print(f'Detected {len(detections)} candidate sources.')

---
## 7 · Detection Results & MCI Diagram

In [ ]:
try:
    fig_mci = plot_mci_diagram(detections)
    plt.show()
except Exception as e:
    print(f'MCI diagram skipped: {e}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
vmin, vmax = np.percentile(injected_image, [1, 99])
ax.imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)

det_x = [d['x'] for d in detections]
det_y = [d['y'] for d in detections]
inj_x = [info['x'] for info in injection_info]
inj_y = [info['y'] for info in injection_info]

ax.scatter(inj_x, inj_y, s=80, facecolors='none', edgecolors='lime',
           lw=1.2, label='Injected')
ax.scatter(det_x, det_y, s=40, facecolors='none', edgecolors='red',
           lw=1.0, marker='s', label='Detected')
ax.legend(loc='upper right', fontsize=11)
ax.set_title('Injected vs Detected Sources')
plt.show()

---
## 8 · Match Detections to Injections (Retrieval)

In [ ]:
MATCH_RADIUS = 5.0  # pixels

retrieval = ClusterRetrieval(injection_info, detections)
retrieval.match_detections(match_radius=MATCH_RADIUS)

stats = retrieval.get_summary_statistics()

print('=== Retrieval Summary ===')
print(f"  Injected         : {stats['n_injected']}")
print(f"  Recovered        : {stats['n_recovered']}")
print(f"  Missed           : {stats['n_missed']}")
print(f"  Completeness     : {stats['overall_completeness']:.1%}")
print(f"  50% mag limit    : {stats['mag_50_limit']:.2f}")

---
## 9 · Completeness Curves

In [ ]:
# --- 9a  Completeness vs magnitude ---
mag_bins = np.linspace(config.cluster_config.mag_min,
                       config.cluster_config.mag_max, 15)

completeness_by_mag = retrieval.completeness_vs_magnitude(mag_bins)

fig, ax = plt.subplots(figsize=(8, 5))
bin_centers = 0.5 * (mag_bins[:-1] + mag_bins[1:])
ax.step(bin_centers, completeness_by_mag, where='mid', lw=2, color='royalblue')
ax.axhline(0.5, ls='--', color='gray', label='50 %')
ax.axhline(0.9, ls=':', color='gray', label='90 %')
ax.set_xlabel('Injected Magnitude', fontsize=13)
ax.set_ylabel('Completeness', fontsize=13)
ax.set_ylim(-0.05, 1.05)
ax.set_title('Completeness vs Magnitude')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'completeness_vs_mag.png'), dpi=150)
plt.show()

In [ ]:
# --- 9b  Completeness vs half-light radius ---
rh_bins = np.linspace(config.cluster_config.r_half_min,
                      config.cluster_config.r_half_max, 10)

completeness_by_rh = retrieval.completeness_vs_size(rh_bins)

fig, ax = plt.subplots(figsize=(8, 5))
bin_centers_rh = 0.5 * (rh_bins[:-1] + rh_bins[1:])
ax.step(bin_centers_rh, completeness_by_rh, where='mid', lw=2, color='darkorange')
ax.axhline(0.5, ls='--', color='gray', label='50 %')
ax.set_xlabel('Half-light Radius (pixels)', fontsize=13)
ax.set_ylabel('Completeness', fontsize=13)
ax.set_ylim(-0.05, 1.05)
ax.set_title('Completeness vs Size')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'completeness_vs_size.png'), dpi=150)
plt.show()

In [ ]:
# --- 9c  2-D completeness map (magnitude × size) ---
completeness_2d = retrieval.completeness_2d(mag_bins, rh_bins)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(mag_bins, rh_bins, completeness_2d.T,
                   cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Completeness')
ax.set_xlabel('Magnitude', fontsize=13)
ax.set_ylabel('Half-light Radius (pixels)', fontsize=13)
ax.set_title('2-D Completeness Map')
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'completeness_2d.png'), dpi=150)
plt.show()

---
## 10 · Save Results

In [ ]:
# Attach our results back to the pipeline object so save_results works
pipe.injected_image = injected_image
pipe.injection_info = injection_info
pipe.retrieval = retrieval
pipe.detection_catalog = detections

try:
    pipe.save_results()
    print(f'All outputs saved to: {config.output_dir}')
except Exception as e:
    print(f'save_results issue: {e}')
    # Manual fallback: save catalog as CSV
    import pandas as pd
    df = pd.DataFrame(injection_info)
    df.drop(columns=['stamp'], errors='ignore', inplace=True)
    outpath = os.path.join(config.output_dir, 'injection_catalog.csv')
    df.to_csv(outpath, index=False)
    print(f'Saved injection catalog to {outpath}')

print('\nDone!')